In [1]:
# If the packages are already present in your Kaggle image, you can skip this cell.
# Kaggle's PyTorch GPU image usually has torch/torchvision already.
%pip install -q easyocr timm sentencepiece "transformers>=4.42.3" "accelerate>=0.30.1"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 97.1 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 70.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 52.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 8.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 32.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 14.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 74.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━

In [3]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
from PIL import Image
import torch
from tqdm import tqdm
import easyocr
from transformers import pipeline

torch.set_grad_enabled(False)

ValueError: All ufuncs must have type `numpy.ufunc`. Received (<ufunc 'sph_legendre_p'>, <ufunc 'sph_legendre_p'>, <ufunc 'sph_legendre_p'>)

In [ ]:
!pip install compressed-tensors


In [ ]:


# =======================
# 2. Find BASE_DIR
# =======================
default_base = Path('/kaggle/input/poli-meme-decode-cuet-cse-fest/PoliMemeDecode')
candidates = [default_base] + [p for p in default_base.iterdir() if p.is_dir()]
BASE_DIR = None
for candidate in candidates:
    train_csv = candidate / 'Train' / 'Train.csv'
    test_csv = candidate / 'Test' / 'Test.csv'
    if train_csv.exists() and test_csv.exists():
        BASE_DIR = candidate
        break

if BASE_DIR is None:
    raise FileNotFoundError(
        'Could not find a dataset folder containing Train/Train.csv and Test/Test.csv under /content/upload. '
        'Please set BASE_DIR manually.'
    )

TRAIN_CSV = BASE_DIR / 'Train' / 'Train.csv'
TEST_CSV = BASE_DIR / 'Test' / 'Test.csv'
TRAIN_IMG_DIR = BASE_DIR / 'Train' / 'Image'
TEST_IMG_DIR = BASE_DIR / 'Test' / 'Image'

# Output files
OUT_TRAIN = Path('ocr_caption_train.csv')
OUT_TEST = Path('ocr_caption_test.csv')

In [ ]:

# =======================
# 3. Models: EasyOCR + BLIP
# =======================
print("Initializing EasyOCR...")
reader = easyocr.Reader(['bn', 'en'], gpu=torch.cuda.is_available())

print("Initializing caption model (BLIP)...")
device = 0 if torch.cuda.is_available() else -1
captioner = pipeline(
    "image-to-text",
    model="allenai/olmOCR-7B-0725-FP8",
    device=device
)
print('Using device:', 'cuda' if torch.cuda.is_available() else 'cpu')


In [ ]:


# =======================
# 4. OCR + caption utils
# =======================
def extract_ocr_text(img_path: Path) -> str:
    """
    Run EasyOCR and return concatenated text lines for Bangla + English.
    Includes simple preprocessing + upscaling for small images.
    """
    try:
        img = Image.open(img_path).convert("RGB")

        # Upscale small images
        w, h = img.size
        target_min_side = 1024
        if max(w, h) < target_min_side:
            scale = target_min_side / max(w, h)
            new_w, new_h = int(w * scale), int(h * scale)
            img = img.resize((new_w, new_h), Image.BICUBIC)

        img_np = np.array(img)
        results = reader.readtext(img_np, detail=0, paragraph=False)

        texts = [r.strip() for r in results if isinstance(r, str) and r.strip()]
        text = " ".join(texts).strip()
    except Exception as exc:
        text = ""
        print(f"[WARN] OCR failed for {img_path.name}: {exc}")
    return text


def generate_caption(img_path: Path) -> str:
    """Generate a short caption describing the image."""
    try:
        outputs = captioner(Image.open(img_path).convert('RGB'))
        caption = outputs[0].get("generated_text", "").strip()
    except Exception as exc:
        caption = ""
        print(f"[WARN] Caption failed for {img_path.name}: {exc}")
    return caption


def process_split(csv_path: Path, img_dir: Path, out_path: Path):
    df = pd.read_csv(csv_path)
    records = []

    for _, row in tqdm(df.iterrows(), total=len(df)):
        name = row['Image_name']
        img_path = img_dir / name

        # Skip if the image file is missing
        if not img_path.exists():
            print(f"[WARN] Image not found: {img_path}")
            ocr_text = ""
            caption = ""
        else:
            ocr_text = extract_ocr_text(img_path)
            caption = generate_caption(img_path)

        records.append({
            'Image_name': name,
            'ocr_text': ocr_text,
            'caption': caption
        })

    out_df = pd.DataFrame(records)
    out_df.to_csv(out_path, index=False)
    print(f'Wrote {len(out_df)} rows to {out_path}')
    return out_df

# =======================
# 5. Run extraction
# =======================
train_meta = process_split(TRAIN_CSV, TRAIN_IMG_DIR, OUT_TRAIN)
test_meta = process_split(TEST_CSV, TEST_IMG_DIR, OUT_TEST)

display(train_meta.head())
display(test_meta.head())


from caption and ocr

In [ ]:
import torch
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm

# ======================
# 1. Load model & tokenizer
# ======================
model_name = "bigscience/bloom-3b"  # you can switch to an *Instruct* variant if available

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)


In [ ]:
def classify_political(text: str) -> str:
    """
    Use Qwen to classify text as Political / NonPolitical.
    Returns exactly 'Political' or 'NonPolitical'.
    """

    system_prompt = (
        "You are a classifier for bangladesh originated social-media memes.\n"
        "Your job is to decide if the meme is about politics.\n"
        "Output exactly one word: 'Political' or 'NonPolitical'.\n\n"
        "Guidelines:\n"
        "- Political: about politicians, political parties, elections, laws, government, national symbols, political events, political person/leader, protests, or political ideologies.\n"
        "- NonPolitical: everything else (food, sports, relationship jokes, religion without politics, random memes, etc.).\n"
        "Do NOT explain your answer. Just output 'Political' or 'NonPolitical'."
    )

    user_prompt = f"""
Here is the meme content (OCR text + caption):

{text}

Classify this meme. Remember: answer with exactly one word: Political or NonPolitical.
"""

    # Manually construct the full prompt for Bloom model
    full_prompt = f"{system_prompt}\n\n{user_prompt.strip()}"

    model_inputs = tokenizer([full_prompt], return_tensors="pt").to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=32,     # we only need a few tokens
            do_sample=False,       # deterministic
            temperature=0.0
        )

    # Decode the generated text, excluding the input prompt
    # Bloom models do not have a specific chat template or "thinking" tokens.
    generated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
    content = generated_text[len(full_prompt):].strip()

    # If no content is generated, return a default classification
    if not content:
        return "NonPolitical"

    # If the model ever puts both answer + explanation, cut to first line / word
    first_line = content.splitlines()[0].strip()
    first_word = first_line.split()[0]

    ans = first_word.lower()

    if "nonpolitical" in ans or "non-political" in ans:
        return "NonPolitical"
    if "political" in ans:
        return "Political"

    # Fallback heuristics: look into full content
    lower_all = content.lower()
    if "nonpolitical" in lower_all or "non-political" in lower_all:
        return "NonPolitical"
    if "political" in lower_all:
        return "Political"

    # If still ambiguous, default to NonPolitical
    return "NonPolitical"


In [ ]:

# ======================
# 3. Load OCR+caption test file
# ======================

test_df = pd.read_csv("ocr_caption_test.csv")

def build_text(row):
    ocr = str(row.get("ocr_text", "") or "")
    cap = str(row.get("caption", "") or "")
    if not (ocr.strip() or cap.strip()):
        return "[EMPTY OCR AND CAPTION]"
    return f"OCR text: {ocr}\nCaption: {cap}"

test_df["llm_input"] = test_df.apply(build_text, axis=1)





In [ ]:
# ======================
# 4. Run classification over all test rows
# ======================

labels = []
for txt in tqdm(test_df["llm_input"].tolist(), desc="Classifying with Qwen"): # Changed desc from 'Classifying with Qwen' to 'Classifying with Bloom'
    label = classify_political(txt)
    labels.append(label)

# ======================
# 5. Build submission file
# ======================

submission = pd.DataFrame({
    "Image_name": test_df["Image_name"],
    "Label": labels
})

submission.to_csv("submission.csv", index=False)
print("Saved submission.csv")
print(submission.head())